In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 0 – IMPORTS
# Loads all required libraries:
#   - xml.etree.ElementTree: parses TEI XML files produced by GROBID
#   - re: used for regex-based text cleaning (e.g. removing extra spaces)
#   - os / glob: scans a folder and collects all .xml file paths
#   - pandas: stores extracted data in tabular DataFrames
#   - BeautifulSoup (bs4): alternative XML parser used in the references extractor
#   - typing: provides type hints (Dict, List, Optional, etc.) for readability
# ─────────────────────────────────────────────────────────────

import xml.etree.ElementTree as ET
import re
from typing import Dict, Union

import xml.etree.ElementTree as ET
from typing import Dict, List, Any, Optional

import xml.etree.ElementTree as ET
import re
from typing import Dict, Union, List, Any, Optional
import os
import pandas as pd
from glob import glob


import pandas as pd
from bs4 import BeautifulSoup
import re
from typing import List, Dict, Optional

import pandas as pd
import xml.etree.ElementTree as ET
from typing import Optional


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 1 – HELPER: extract_filename(text)
# Extracts the paper ID (filename without extension) from a full file path.
# Example: '/content/smith2020.pdf.tei.xml'  →  'smith2020'
# This ID is used as a unique key to link rows across the three output tables.
# ─────────────────────────────────────────────────────────────

def extract_filename(text):
    match = re.search(r"/content/(.+?)\.pdf\.tei\.xml", text)
    return match.group(1) if match else None


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2 – FUNCTION: extract_sections_content(file_path)
#
# PURPOSE: Extracts the body text of a paper, organised by section.
#
# INPUT:  Path to a single GROBID-generated .tei.xml file.
# OUTPUT: DataFrame with columns → Paper_Id | Title | Section | Content
#
# WHAT IT DOES STEP BY STEP:
#   1. Parses the XML and reads the paper's main title from <titleStmt>.
#   2. Skips everything before the 'Introduction' section (e.g. abstract).
#   3. Walks through each <div> in <body> and collects paragraph text.
#   4. Replaces inline citation tags (e.g. <ref target="#b23">[24]</ref>)
#      with the reference ID (@b23) so citations remain traceable in plain text.
#   5. Merges subsections (e.g. 2.1, 2.2) into their parent section (2)
#      to keep one row per top-level section.
#   6. Replaces '#' with '@' in content (avoids issues in downstream tools).
#   7. Replaces double newlines with '[LINE BREAK]. ' for cleaner Excel output.
#   8. Prefixes each section title with a counter: 'Section 1: Introduction'.
# ─────────────────────────────────────────────────────────────

def extract_sections_content(file_path: str) -> pd.DataFrame:
    """
    Extracts the paper title, then groups sections, processes citations, and formats content into a DataFrame.

    This function parses a TEI XML file and performs six main tasks:
    1.  Extracts the main title of the paper.
    2.  Extracts sections starting from 'Introduction'.
    3.  Merges subsections (e.g., '2.1', '2.2') under their main parent section.
    4.  Replaces inline citation text (e.g., '[24]') with the target ID from the
        <ref> tag's 'target' attribute (e.g., '#b23').
    5.  Formats the final section titles by prepending a sequential number,
        e.g., "Section 1: Introduction".
    6.  Replaces double newlines ("\n\n") with "[LINE BREAK]. " in the final
        text content for better readability in certain contexts.

    Args:
        file_path: The path to the TEI XML file.

    Returns:
        A pandas DataFrame with 'Title', 'Section', and 'Content' columns.
        Returns an empty DataFrame if processing fails.
    """
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
    except ET.ParseError as e:
        print(f"Error: Could not parse XML file. Details: {e}")
        return pd.DataFrame()
    except FileNotFoundError:
        print(f"Error: File not found at '{file_path}'")
        return pd.DataFrame()

    # Define the TEI namespace for XPath queries
    namespace = {'tei': 'http://www.tei-c.org/ns/1.0'}
    if '}' in root.tag:
        ns_uri = root.tag.split('}')[0][1:]
        namespace['tei'] = ns_uri

    # Extract the paper title from the <teiHeader>
    paper_title = "Title Not Found"
    try:
        title_element = root.find('.//tei:titleStmt/tei:title[@level="a"][@type="main"]', namespace)
        if title_element is not None:
            paper_title = re.sub(r'\s+', ' ', "".join(title_element.itertext())).strip()
    except Exception as e:
        print(f"Warning: Could not extract paper title. Details: {e}")

    def ns_tag(tag_name):
        return f"{{{namespace['tei']}}}{tag_name}"

    body = root.find('tei:text/tei:body', namespace)
    if body is None:
        print("Error: Could not find the <body> tag in the XML file.")
        return pd.DataFrame()

    divs = body.findall('tei:div', namespace)
    if not divs:
        print("Error: No <div> sections found in the body.")
        return pd.DataFrame()

    extracted_data = {}
    main_section_titles = {}
    is_extracting = False

    for div in divs:
        head_element = div.find('tei:head', namespace)
        if head_element is None:
            continue

        section_title = re.sub(r'\s+', ' ', "".join(head_element.itertext())).strip()

        # Start extraction when 'Introduction' section is found
        if not is_extracting and 'introduction' in section_title.lower():
            is_extracting = True

        if is_extracting:
            paragraphs = []
            for p_element in div.findall('tei:p', namespace):
                # Reconstruct paragraph text, replacing <ref> citation tags with their target IDs
                parts = []
                if p_element.text:
                    parts.append(p_element.text)

                for child in p_element:
                    if child.tag == ns_tag('ref'):
                        target = child.get('target', '')
                        if target:
                            parts.append(f" {target}")
                    else:
                        if child.text:
                            parts.append(child.text)

                    if child.tail:
                        parts.append(child.tail)

                full_paragraph = "".join(parts)
                cleaned_paragraph = re.sub(r'\s+', ' ', full_paragraph).strip()
                if cleaned_paragraph:
                    paragraphs.append(cleaned_paragraph)

            section_content = "\n\n".join(paragraphs)

            # Replace '#' with '@' to avoid formatting issues in downstream tools
            section_content = section_content.replace('#', '@')

            if not section_content:
                continue

            # Merge subsections (e.g. 2.1, 2.2) under their top-level parent section (2)
            n_attr = head_element.get('n', '').strip()
            if not n_attr:
                extracted_data[section_title] = section_content
                continue

            main_section_number = n_attr.split('.')[0]
            if main_section_number not in main_section_titles:
                main_section_titles[main_section_number] = section_title
                extracted_data[section_title] = section_content
            else:
                parent_title = main_section_titles[main_section_number]
                extracted_data[parent_title] += "\n\n" + section_content

    if not extracted_data:
        print(extract_filename(file_path),"Error: Could not find an 'Introduction' section to start extraction.")
        return pd.DataFrame()

    data_for_df = []
    section_counter = 1
    paper_id = extract_filename(file_path)

    for title, text in extracted_data.items():
        # Label each section sequentially: 'Section 1: Introduction', 'Section 2: Methods', etc.
        formatted_title = f"Section {section_counter}: {title}"

        # Replace double newlines with a readable placeholder for Excel cells
        processed_text = text.replace("\n\n", "[LINE BREAK]. ")

        data_for_df.append((paper_id, paper_title, formatted_title, processed_text))
        section_counter += 1

    df = pd.DataFrame(data_for_df, columns=["Paper_Id", 'Title', 'Section', 'Content'])

    return df


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3 – FUNCTIONS: text_or_default() + get_publication_info()
#
# text_or_default(elem, default):
#   Small utility — safely reads an XML element's text.
#   Returns the stripped text if present, or 'default' if the element
#   is missing or empty. Avoids repetitive None-checks throughout the parser.
#
# get_publication_info(file_path):
#   PURPOSE: Extracts front-matter metadata from a single paper.
#   INPUT:   Path to a GROBID .tei.xml file.
#   OUTPUT:  One-row DataFrame with columns:
#              Paper_Id | Title | Abstract | Authors | Affiliations | Year
#
#   WHAT IT DOES:
#     - Title   → reads <titleStmt><title level='a' type='main'>
#     - Year    → reads the 'when' attribute of <date type='published'>
#     - Abstract→ reads the first <p> inside <abstract>
#     - Authors → loops over every <author> tag, concatenates first name + surname
#     - Affiliations → for each author reads <orgName> and <address> fields,
#                      deduplicates them, and joins with '; '
# ─────────────────────────────────────────────────────────────

def text_or_default(elem, default=""):
    """Return elem.text.strip() if present and non-empty; otherwise default."""
    if elem is not None:
        txt = elem.text
        if isinstance(txt, str):
            s = txt.strip()
            if s:
                return s
    return default

def get_publication_info(file_path: str) -> Optional[pd.DataFrame]:
    """
    Robust TEI parser → 1-row DataFrame, or None on error.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            xml_content = f.read()

        ns = {'tei': 'http://www.tei-c.org/ns/1.0'}
        root = ET.fromstring(xml_content)

        def extract_filename(p):  # local fallback: returns filename without extension
            import os
            return os.path.splitext(os.path.basename(p))[0]

        paper_id = extract_filename(file_path)

        # Extract the paper's main title
        title_elem = root.find('.//tei:titleStmt/tei:title[@level="a"][@type="main"]', ns)
        title = text_or_default(title_elem, "Not Found")

        # Extract publication year from the 'when' attribute (format: YYYY or YYYY-MM-DD)
        date_elem = root.find('.//tei:publicationStmt/tei:date[@type="published"]', ns)
        if date_elem is not None and 'when' in date_elem.attrib and date_elem.attrib['when']:
            year_str = date_elem.attrib['when'][:4]
            try:
                year = int(year_str)
            except (ValueError, TypeError):
                year = "Not Found"
        else:
            year = "Not Found"

        # Extract the abstract (first paragraph only)
        abs_p = root.find('.//tei:profileDesc/tei:abstract/tei:div/tei:p', ns)
        abstract = text_or_default(abs_p, "Not Found")

        # Extract author names and their institutional affiliations
        authors_list = []
        unique_affiliations = set()  # set ensures no duplicate institutions

        for author in root.findall('.//tei:sourceDesc/tei:biblStruct/tei:analytic/tei:author', ns):
            pn = author.find('tei:persName', ns)
            first = text_or_default(pn.find('tei:forename[@type="first"]', ns) if pn is not None else None, "")
            sur = text_or_default(pn.find('tei:surname', ns) if pn is not None else None, "")
            full_name = (f"{first} {sur}").strip()
            if full_name:
                authors_list.append(full_name)

            # Build affiliation string from org names + address fields
            aff = author.find('tei:affiliation', ns)
            if aff is not None:
                parts = []
                for org in aff.findall('.//tei:orgName', ns):
                    t = text_or_default(org, "")
                    if t:
                        parts.append(t)

                addr = aff.find('tei:address', ns)
                if addr is not None:
                    for tag in ['tei:addrLine', 'tei:settlement', 'tei:postCode', 'tei:country']:
                        val = text_or_default(addr.find(tag, ns), "")
                        if val:
                            parts.append(val)

                # Remove duplicates within this affiliation while preserving order
                seen = set()
                unique_parts = []
                for p in parts:
                    if p not in seen:
                        seen.add(p)
                        unique_parts.append(p)

                aff_str = ", ".join(unique_parts).strip()
                if aff_str:
                    unique_affiliations.add(aff_str)

        publication_data = {
            "Paper_Id": paper_id,
            "Title": title,
            "Abstract": abstract,
            "Authors": "; ".join(authors_list),
            "Affiliations": "; ".join(sorted(unique_affiliations)),
            "Year": year,
        }

        return pd.DataFrame([publication_data])

    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return None
    except ET.ParseError as e:
        print(f"Error: Failed to parse XML from '{file_path}'. Details: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4 – FUNCTION: extract_references(filename)
#
# PURPOSE: Extracts the full bibliography (reference list) from a single paper.
# INPUT:   Path to a GROBID .tei.xml file.
# OUTPUT:  DataFrame with columns:
#            Citation_Id | Authors | Year | Title | Source | PaperId | Paper
#
# WHAT IT DOES:
#   1. Reads the XML with BeautifulSoup (more lenient parser than ElementTree).
#   2. Extracts the citing paper's title and ID (for traceability).
#   3. Finds the <div type='references'> block — the bibliography section.
#   4. Loops over every <biblStruct> entry (one per reference) and extracts:
#        - Citation_Id : the reference's xml:id (e.g. 'b23'), prefixed with '@'
#        - Authors     : all author names joined with '; '
#        - Year        : publication year from <date type='published'>
#        - Title       : article title from <analytic>, or book title from <monogr>
#        - Source      : journal name from <monogr><title level='j'>
#   5. Adds the citing paper's title and ID to every row so rows stay linked
#      to their source paper when tables are merged later.
# ─────────────────────────────────────────────────────────────

def extract_references(filename: str) -> pd.DataFrame:
    """
    Parses a TEI XML file to extract the main paper's title and its bibliographic references.

    This function reads an XML file, finds the paper's main title, and then extracts
    the ID, authors, year, title, and source for each <biblStruct> entry in the
    references section. The paper's title is added as a new column to each row.

    Args:
        filename: The path to the TEI XML file.

    Returns:
        A pandas DataFrame with columns: 'Paper', 'id', 'authors', 'year', 'title', 'source'.
        Returns an empty DataFrame if the file is not found.
    """
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            content = f.read()
    except FileNotFoundError:
        print(f"Error: The file '{filename}' was not found.")
        return pd.DataFrame()

    soup = BeautifulSoup(content, 'lxml-xml')

    # Get the paper's short ID (filename without extension)
    paper_id = extract_filename(filename)

    # Get the paper's full title from the header
    paper_title = "Unknown"
    title_stmt = soup.find('titleStmt')
    if title_stmt:
        if title_tag := title_stmt.find('title', {'level': 'a', 'type': 'main'}):
            paper_title = title_tag.text.strip()

    # Locate the references section — returns early if not found
    references_div = soup.find('div', type='references')
    if not references_div:
        print("Warning: <div type='references'> not found in the XML file.")
        return pd.DataFrame({'Paper': [paper_title]})

    bibl_structs = references_div.find_all('biblStruct')

    references_list: List[Dict[str, Optional[str]]] = []

    for bibl in bibl_structs:
        ref_data = {'Citation_Id': None, 'Authors': None, 'Year': None, 'Paper_Id': None, 'Title': None, 'Source': None}

        # Reference ID (e.g. 'b23') — will be prefixed with '@' at the end
        ref_data['Citation_Id'] = bibl.get('xml:id')

        # Build author list: 'Forename Surname' joined with '; '
        authors_tags = bibl.find_all('author')
        authors = []
        for author_tag in authors_tags:
            pers_name = author_tag.find('persName')
            if pers_name:
                surnames = " ".join([s.text for s in pers_name.find_all('surname')])
                forenames = " ".join([f.text for f in pers_name.find_all('forename')])
                authors.append(f"{forenames} {surnames}".strip())
        ref_data['Authors'] = "; ".join(authors) if authors else None

        # Title: prefer <analytic> (journal article), fall back to <monogr> (book/chapter)
        analytic_tag = bibl.find('analytic')
        monogr_tag = bibl.find('monogr')

        ref_data['Paper_Id'] = paper_id

        if analytic_tag and (title_tag := analytic_tag.find('title', {'level': 'a', 'type': 'main'})):
            ref_data['Title'] = title_tag.text.strip()
        elif monogr_tag and (title_tag := monogr_tag.find('title', {'level': 'm', 'type': 'main'})):
            ref_data['Title'] = title_tag.text.strip()

        # Year and source journal come from <monogr>
        if monogr_tag:
            if date_tag := monogr_tag.find('date', {'type': 'published'}):
                if date_tag.has_attr('when'):
                    if year_match := re.search(r'\d{4}', date_tag['when']):
                        ref_data['Year'] = year_match.group(0)
                if not ref_data['Year']:
                    if year_match := re.search(r'\d{4}', date_tag.text.strip()):
                        ref_data['Year'] = year_match.group(0)

            # Journal name (level='j' means journal in TEI)
            if journal_title_tag := monogr_tag.find('title', {'level': 'j'}):
                ref_data['Source'] = journal_title_tag.text.strip()

        references_list.append(ref_data)

    if not references_list:
        return pd.DataFrame({'Paper': [paper_title]})

    df = pd.DataFrame(references_list)

    # Add citing paper info to every row for cross-table traceability
    df['Paper'] = paper_title
    df['PaperId'] = paper_id

    # Prefix Citation_Id with '@' to match the format used in the section content table
    if 'Citation_Id' in df.columns:
        df['Citation_Id'] = df['Citation_Id'].apply(lambda x: f'@{x}' if pd.notna(x) else None)

    column_order = ['Citation_Id', 'Authors', 'Year', 'Title', 'Source', 'PaperId', 'Paper']
    final_columns = [col for col in column_order if col in df.columns]

    return df[final_columns]


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5 – SCAN INPUT FOLDER
# Sets the folder that contains all GROBID-processed XML files.
# glob() returns a list of every .xml file path in that folder.
# ⚠️  Change xml_directory to match where your .tei.xml files are stored.
# ─────────────────────────────────────────────────────────────

xml_directory = "/content/"  # Change to your actual directory
xml_files = glob(os.path.join(xml_directory, "*.xml"))
xml_files  # Displays the list of found files — verify this before running the next cells


['/content/paper7.pdf.tei.xml',
 '/content/paper3.pdf.tei.xml',
 '/content/paper2.pdf.tei.xml',
 '/content/paper6.pdf.tei.xml',
 '/content/paper4.pdf.tei.xml',
 '/content/paper1.pdf.tei.xml',
 '/content/paper5.pdf.tei.xml']

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6 – EXTRACT SECTION CONTENT FROM ALL PAPERS
# Loops over every XML file and calls extract_sections_content().
# Each call returns a DataFrame with one row per section.
# All DataFrames are concatenated into df_section_content.
# Result: one table with every section of every paper.
# ─────────────────────────────────────────────────────────────

all_data = []

for xml_file in xml_files:
    section_content = extract_sections_content(xml_file)
    all_data.append(section_content)

df_section_content = pd.concat(all_data, ignore_index=True)
df_section_content  # Preview the table


,Paper_Id,Title,Section,Content
0,paper7,Why do some retracted articles continue to get...,Section 1: Introduction,Retractions of scientific publications are oft...
1,paper7,Why do some retracted articles continue to get...,Section 2: Data and sampling strategy,This study is based on a corpus of retracted p...
2,paper7,Why do some retracted articles continue to get...,Section 3: Methods,"Three main methods are used in this paper, whi..."
3,paper7,Why do some retracted articles continue to get...,Section 4: Results,The presentation of the results follows the st...
4,paper7,Why do some retracted articles continue to get...,Section 5: Network approach: retracted publica...,The citation dynamics of these publications ar...
...,...,...,...,...
85,paper5,Continued post-retraction citation of a fraudu...,Section 21: Retraction notice in full-text sites,We found the retraction notice in only two ful...
86,paper5,Continued post-retraction citation of a fraudu...,Section 22: Database records for the retracted...,For each database record referring to the arti...
87,paper5,Continued post-retraction citation of a fraudu...,Section 23: Database records for the retractio...,The most problematic results came in attemptin...
88,paper5,Continued post-retraction citation of a fraudu...,Section 24: Table 8,Database records for the retraction notice @b7...


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 7 – EXTRACT PUBLICATION METADATA FROM ALL PAPERS
# Loops over every XML file and calls get_publication_info().
# Each call returns a one-row DataFrame with title, authors,
# abstract, affiliations, and year.
# All rows are concatenated into df_publication_info.
# Result: one table with one row per paper (the 'overview' table).
# ─────────────────────────────────────────────────────────────

all_data = []

for xml_file in xml_files:
    paper_content = get_publication_info(xml_file)
    all_data.append(paper_content)

df_publication_info = pd.concat(all_data, ignore_index=True)
df_publication_info  # Preview the table


,Paper_Id,Title,Abstract,Authors,Affiliations,Year
0,paper7.pdf.tei,Why do some retracted articles continue to get...,Retracted publications can still receive a sub...,Marion Schmidt,German Center for Higher Education Research an...,2024
1,paper3.pdf.tei,What difference might retractions make? An est...,The extent to which a retraction might require...,Daniele Fanelli; Julie Wong; David Moher,"Centre for Journalology, Clinical Epidemiology...",2021
2,paper2.pdf.tei,Addressing Unreliability Propagation in Scient...,Today's scientists rely on scientific artifact...,Heng Zheng; Yuanxi Fu; M Sarol; Ishita Sarraf;...,"Grinnell College Grinnell, USA; Jodi Schneider...",2024
3,paper6.pdf.tei,Critical citations in knowledge construction a...,Critical citations are lacking a common defini...,Frederique Bordignon,"Ecole des Ponts, Marne-la-Vallée, France",2022
4,paper4.pdf.tei,Continued use of retracted papers: Temporal tr...,We present the first database-wide study on th...,Tzu-Kun Hsiao; Jodi Schneider,"School of Information Sciences, University of ...",2021
5,paper1.pdf.tei,Towards Knowledge Maintenance in Scientific Di...,Scientific digital libraries speed disseminati...,Yuanxi Fu; Jodi Schneider,"School of Information Sciences, University of ...",2020
6,paper5.pdf.tei,Continued post-retraction citation of a fraudu...,This paper presents a case study of long-term ...,Jodi Schneider; Di Ye; Alison Hill; Ashley Whi...,"Data Science Institute, Columbia University, N...",2020


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 8 – EXTRACT REFERENCE LISTS FROM ALL PAPERS
# Loops over every XML file and calls extract_references().
# Each call returns a DataFrame with one row per cited reference.
# All DataFrames are concatenated into df_references.
# Result: one table with every citation from every paper.
# ─────────────────────────────────────────────────────────────

all_data = []

for xml_file in xml_files:
    references_info = extract_references(xml_file)
    all_data.append(references_info)

df_references = pd.concat(all_data, ignore_index=True)
df_references  # Preview the table


,Citation_Id,Authors,Year,Title,Source,PaperId,Paper
0,@b0,J L Austin,2009,How to do things with words: The William James...,None,paper7,Why do some retracted articles continue to get...
1,@b1,P Azoulay; J Furman; J Krieger; F Murray,2012,None,None,paper7,Why do some retracted articles continue to get...
2,@b2,J Bar-Ilan; G Halevi,2017,Post retraction citations in context: A case s...,Scientometrics,paper7,Why do some retracted articles continue to get...
3,@b3,S D Bolboacă; D.-V Buhai; M Aluaș; A E Bulboacă,2019,Post retraction citations among manuscripts re...,PLoS ONE,paper7,Why do some retracted articles continue to get...
4,@b4,H Bornemann-Cimenti; I S Szilagyi; A Sandner-K...,2016,Perpetuation of retracted publications using t...,Science and Engineering Ethics,paper7,Why do some retracted articles continue to get...
...,...,...,...,...,...,...,...
417,@b119,K B Wray; L E Andersen,2018,Retractions in science,Scientometrics,paper5,Continued post-retraction citation of a fraudu...
418,@b120,N D Wright,1991,None,None,paper5,Continued post-retraction citation of a fraudu...
419,@b121,K Wright; C Mcdaid,2011,Reporting of article retractions in bibliograp...,Journal of the Medical Library Association: JMLA,paper5,Continued post-retraction citation of a fraudu...
420,@b122,D Ye; A Hill; A Whitehorn; J Schneider,2005,Citation context annotation for new and newly ...,None,paper5,Continued post-retraction citation of a fraudu...


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 9 – EXPORT TO EXCEL
# Saves the three DataFrames to separate .xlsx files.
# File names include the expert's name for easy identification.
#
#   Test_<expert>_Overallv01.xlsx   → one row per paper (metadata)
#   Test_<expert>_Sectionsv01.xlsx  → one row per section (body text)
#   Test_<expert>_Referencesv01.xlsx→ one row per cited reference
#
# ⚠️  Change the 'expert' variable to your own name before running.
# ─────────────────────────────────────────────────────────────

expert = "Jodi"  # Change to your name or identifier

df_publication_info.to_excel(f"Test_{expert}_Overallv01.xlsx", index=False)
df_section_content.to_excel(f"Test_{expert}_Sectionsv01.xlsx", index=False)
df_references.to_excel(f"Test_{expert}_Referencesv01.xlsx", index=False)
